In [10]:
import math
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

In [31]:
class Value():
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op
        self._backward = lambda: None
        self.label = label

    def __repr__(self):
        return f'(label={self.label}, data={self.data:.4f}, grad={self.grad:.4f})'

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), _op='+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), _op='*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out
    
    def __rmul__(self, other):
        return self * other
    
    def __pow__(self, other):
        if not isinstance(other, (int, float)):
            raise TypeError("Only supporting int/float powers")

        out = Value(self.data ** other, (self, ), f'**{other}')

        def _backward():
            self.grad += other * out.grad * (self.data ** (other - 1))
        
        out._backward = _backward

        return out
    
    def __truediv__(self, other):
        return self * (other ** -1)

    def exp(self):
        data = math.exp(self.data)
        out = Value(data, (self, ), 'exp')

        def _backward():
            self.grad += out.grad * data
        
        out._backward = _backward

        return out

    def tanh(self):
        data = self.data
        t = (math.exp(2 * data) - 1)/(math.exp(2 * data) + 1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward 

        return out

    def backward(self):

        topo = []
        visited = set()

        def build_topo(root):
            if root not in visited:
                visited.add(root)
                for node in root._prev:
                    build_topo(node)
                topo.append(root)
        
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

In [32]:
a = Value(2)
b = Value(3)
L = a / b

In [33]:
L.backward()

In [41]:
list(L._prev)[1]

(label=, data=0.3333, grad=2.0000)

In [52]:
w1 = Value(-3, label='w1')
w2 = Value(1, label='w2')

x1 = Value(2, label='x1')
x2 = Value(0, label='x2')

b = Value(6.8813735870195432, label='b')

w1x1 = w1 * x1
w1x1.label = 'w1x1'

w2x2 = w2 * x2
w2x2.label = 'w2x2'

w1x1w2x2 = w1x1 + w2x2
w1x1w2x2.label = 'w1x1w2x2'

n = w1x1w2x2 + b
n.label = 'n'

o = n.tanh()
o.label = 'o'


In [53]:
o

(label=o, data=0.7071, grad=0.0000)

In [54]:
o.backward()

In [61]:
x1

(label=x1, data=2.0000, grad=-1.5000)